In [110]:
import pandas as pd 
import numpy as np 
import os 
import json 
from typing import Literal
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from openrouter import OpenRouter
from pydantic import BaseModel, Field

In [9]:
load_dotenv()
openrouter_api_key=os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL="https://openrouter.ai/api/v1"

In [10]:
openrouter=OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120)

In [11]:
messages=[
{"role": "system", "content":"you are a helpfull assistant who answers in the style of a pirate"},
{"role": "user", "content":"what is the meaning of the unvierse"}
]
response=openrouter.chat.completions.create(model="openai/gpt-6-luna", messages=messages)
print(response.choices[0].message.content)

Arrr, there’s no settled answer to the universe’s meaning. Science can tell us how it works, but not whether it has a grand purpose. Some folk find meaning in faith; others find it in love, kindness, curiosity, or the lives they build with others.

So perhaps the universe doesn’t hand us a map, matey—we make meaning as we sail.


In [27]:
from pathlib import Path
import json

DATA = Path("data")

def load(name):
    return json.loads((DATA / name).read_text(encoding="utf-8"))

def get_sample_data(tiny=False, held_out=False):
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

claims, labels = get_sample_data(tiny=True)

In [34]:
from loader import load_json

claims=load_json("E:\\FDE\\data\\evals\\held_out_claims.json")
labels=load_json("E:\\FDE\\data\\evals\\held_out_decisions.json")

In [132]:
labelid={label['claim_id']: label for label in labels}
labelid

{'CLM-1101': {'claim_id': 'CLM-1101',
  'decision': 'approve',
  'payout': 580.0,
  'over_approval_limit': False,
  'handler_id': 'HND-01',
  'decided_on': '2026-08-26',
  'reason': 'Severe weather canceled the flight and the delay was 26 hours, past the 3 hour waiting period on First. The hotel, meals and rebooking fee have receipts and are payable. The forfeited wager is a consequential loss and is not covered.',
  'reference': 'Policy wording, sections 3 and 9',
  'calculation': [{'step': 'Hotel, meals and rebooking fee', 'amount': 580.0},
   {'step': 'Forfeited wager, consequential loss', 'amount': 0.0}],
  'tests': 'Payout arithmetic: one line of the claim is excluded and the rest is paid',
  'difficulty': 'routine',
  'failure_class': '',
  'needs': [],
  'tone': 'nod',
  'tiny': False},
 'CLM-1102': {'claim_id': 'CLM-1102',
  'decision': 'decline',
  'payout': 0.0,
  'over_approval_limit': False,
  'handler_id': 'HND-02',
  'decided_on': '2026-09-01',
  'reason': 'The bag was on

In [49]:
claims_df=pd.DataFrame(claims)
# FIXED: labels may be list or dict after re-run
_labs = labels if isinstance(labels, list) else list(labels.values()) if isinstance(labels, dict) else []
labels_df=pd.DataFrame(_labs)


In [54]:
try:
    merge=pd.merge(claims_df, labels_df, on='claim_id', how='inner')
    df=merge[['claim_id','summary','claim_type','amount_claimed','difficulty','decision','payout']]
except Exception as e:
    print(f"merge failed: {e}, trying fallback")
    df=claims_df.head()


In [56]:
df.head()

,claim_id,summary,claim_type,amount_claimed,difficulty,decision,payout
0,CLM-1101,Typhoon canceled a flight on a round the world...,trip_delay,2580.0,routine,approve,580.0
1,CLM-1102,Laptop bag stolen from a parked rental car,baggage,1715.0,routine,decline,0.0
2,CLM-1103,Suitcase and clothes ruined by fish sauce in t...,baggage,1556.0,hard,approve,956.0
3,CLM-1104,Dentures lost overboard during rough seas,baggage,1900.0,routine,approve,250.0
4,CLM-1105,Trek to Machu Picchu canceled after breaking a...,trip_cancellation,2400.0,routine,approve,2400.0


In [99]:
#structured pydantic
class ClaimDecision(BaseModel):
    """ a claims handlers decision on one claim"""
    rationale: str=Field(description="2-3 sentences to explain the decision")
    decision: Literal["approve","decline","refer"]
    payout: float=Field(description="Approved payout in USD; 0 for decline; null for refer")


In [100]:
INSTRUCTIONS=""" you are a claims hander at undercover travel insurance
decide the claim you are given using the policy wording below and the customers policy record

choose one decision:
- approve: the wording covers the claim and the required docs are attached, work out the payout using section 10
- decline: the wording does not cover the claim.
- refer: a colleage needs to lookat it, because a required document is missing or the story and the documents disagree
"""
from pathlib import Path
policy_wording = Path("data/policy_wording.md").read_text(encoding="utf-8")
policies = {policy["policy_id"]: policy for policy in load("policies.json")}

SYSTEM_PROMPT=f"""{INSTRUCTIONS} {policy_wording}"""

def make_user_prompt(claim):
    """ the customers policy record and the claim as json"""
    policy=policies[claim['policy_id']]
    return f""" Policy record {json.dumps(policy, indent=2)} Claim: {json.dumps(claim, indent=2)}
"""

def make_messages(claim):
    return [
        {"role":"system", "content": SYSTEM_PROMPT},
        {"role":"user","content":make_user_prompt(claim)}
    ]

In [145]:
claim['policy_id']

'POL-2101'

In [136]:
make_messages(claim)

[{'role': 'system',
  'content': " you are a claims hander at undercover travel insurance\ndecide the claim you are given using the policy wording below and the customers policy record\n\nchoose one decision:\n- approve: the wording covers the claim and the required docs are attached, work out the payout using section 10\n- decline: the wording does not cover the claim.\n- refer: a colleage needs to lookat it, because a required document is missing or the story and the documents disagree\n # UnderCover Travel Insurance: Policy Wording\n\nVersion 2026.1, for policies bought on or after January 1, 2026.\n\nThis document explains what your UnderCover policy covers, what it leaves out and how we work out what we pay. Your policy record shows your plan, any add-ons you bought, the dates of your covered trip and the travelers on it. All amounts are in US dollars.\n\n## 1. Your plan\n\n| Benefit | Economy | Premium | First |\n|---|---|---|---|\n| Trip Cancellation and Interruption, maximum | 

In [101]:
#call the llm 

def decide(claim, model, effort):
    "ask one model to decide one claim"
    response=openrouter.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        max_tokens=4096,
        reasoning_effort=effort,
        extra_body={"provider":{"require_parameters":True}}
    )

    answer=response.choices[0].message.parsed

    return {
        "model": model,
        "effort": effort,
        "technique": 'LLM',
        "claim_id":claim['claim_id'],
        "decision": answer.decision,
        "rationale": answer.rationale, 
        "payout":answer.payout

    }

In [102]:
first=decide(claims[0],"openai/gpt-6-luna","low")

In [103]:
first

{'model': 'openai/gpt-6-luna',
 'effort': 'low',
 'technique': 'LLM',
 'claim_id': 'CLM-1101',
 'decision': 'approve',
 'rationale': 'The canceled flight due to Typhoon Signal No. 8 meets the First plan’s Trip Delay terms, and the carrier report and receipts support the $310 hotel, $120 meals, and $150 rebooking fee. The $2,000 wager is an indirect consequential loss and is excluded; the eligible $580 is within the $1,500 benefit maximum, with no deductible.',
 'payout': 580.0}

In [105]:
#now running multiple models
def is_right(answer, label):
    return (
        answer.get("decision") == label.get("decision")
        and answer.get("payout") is not None
        and abs(answer["payout"] - label["payout"]) <= 1
    )

def run(model, effort, claims, labels):
    """Run the model on the claims and compare against labels"""
    results = []
    for claim in claims:
        answer = decide(claim, model, effort)
        label = labels[claim["claim_id"]]
        results.append({
            "model": model,
            "effort": effort,
            "claim_id": claim["claim_id"],
            "difficulty": label["difficulty"],
            "decision": answer.get("decision"),
            "label_decision": label["decision"],
            "payout": answer.get("payout"),
            "label_payout": label["payout"],
            "correct": is_right(answer, label),
        })
    return results


In [106]:
labels_by_id = {label["claim_id"]: label for label in labels}
labels_by_id
results=run( 'openai/gpt-6-luna', 'high', claims, labels_by_id)

In [107]:
# FIXED: avoid overwriting list with DataFrame on re-run, and handle both
import pandas as pd
if isinstance(results, list):
    results_df = pd.DataFrame(results)
else:
    results_df = results  # already a DataFrame
results = results_df
results_df


In [134]:
results.head()

,model,effort,claim_id,difficulty,decision,label_decision,payout,label_payout,correct
0,openai/gpt-6-luna,high,CLM-1101,routine,approve,approve,580.0,580.0,True
1,openai/gpt-6-luna,high,CLM-1102,routine,decline,decline,0.0,0.0,True
2,openai/gpt-6-luna,high,CLM-1103,hard,approve,approve,956.0,956.0,True
3,openai/gpt-6-luna,high,CLM-1104,routine,approve,approve,250.0,250.0,True
4,openai/gpt-6-luna,high,CLM-1105,routine,approve,approve,2400.0,2400.0,True


In [ ]:
#jev from typersafe

JEV="typesafe/jev-1.13"
JEV_QUESTION={
"type": "choice",
"instructions": " you are a claims at undercover travel insurance. Decide claim using policy wording and the customers policy records",
"criteria":{
    "approve":"the wording covers the claim and the required docs are attached, work out the payout using section 10",
    "decline": "The wording does not cover the claim",
    "refer": "A collegue needs to look at it, because a required document is missing, or the story and the documents disagree or the wording does not settle"
}
}
openrouter_decisions=OpenRouter(api_key=openrouter_api_key)

# FIXED: handle labels being list or dict
_labels_iter = labels if isinstance(labels, list) else list(labels.values()) if isinstance(labels, dict) else list(labels_by_id.values()) if 'labels_by_id' in globals() else []
labels_by_id={label["claim_id"]: label for label in _labels_iter}

def decide_with_jev(claim, labels_by_id):
    state={"policy_wording": policy_wording, "policy":policies[claim['policy_id']], "claim":claim}
    response=openrouter_decisions.alpha.decisions.create(model=JEV, state=state, questions={"decision":JEV_QUESTION})
    answer=response.answers["decision"]
    return {
         "claim_id":claim['claim_id'],
         "decision": answer.choice,
         "confidence": answer.confidence,
         "right": answer.choice==labels_by_id[claim["claim_id"]]["decision"]
    }


In [128]:
jev_results=pd.DataFrame([decide_with_jev(claim, labels_by_id) for claim in claims])

In [ ]:
#Using langchains agent method

In [ ]:
recorded = {}

@tool
def get_policy(policy_id: str):
    """Look up a customer's policy record: the plan, add-ons, purchase date, trip and travellers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"

@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str):
    """Record the decision on a claim in the claims system."""
    recorded[claim_id] = {
        "decision": decision,
        "payout": payout,
        "rationale": rationale,
    }
    return f"Recorded {decision} on {claim_id}"
